# CMPE 58T — 07 Combined System
## A2b (BiLSTM + BERTurk + POS) + UD FEATS Morphological Injection

**Motivation:** A2b achieved the best single-system LAS (74.73) and Morph V2 showed that UD FEATS injection provides a large LAS gain (+9.83 over baseline, *** significant). Combining them injects morphological features into the best-performing architectural configuration.

**Architecture:**
- Base: A2b checkpoint (BiLSTM on top of BERTurk, POS input features)
- Addition: UD FEATS MorphEmbedder (dim=32, sum pooling) + MorphProjection
- Injection point: after `parser.model.encode()` — i.e. after BiLSTM output, before biaffine MLPs
- The morph patch works identically for LSTM-encoder and BERT-encoder modes

**Training:** Fine-tunes from A2b checkpoint using Adam + ExponentialLR (same optimizer as A2b in 03_ablation). Only morph modules + biaffine head are updated if desired, or all parameters jointly.

**Prerequisites:** `03_ablation.ipynb` must have produced `a2b_lstm_with_pos.pt`.

---
## 0 — Mount Drive & Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

PROJECT_DIR  = "/content/drive/MyDrive/58T - Application Project"
DATA_DIR     = os.path.join(PROJECT_DIR, "data")
VOCAB_DIR    = os.path.join(PROJECT_DIR, "vocab")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
ABLATION_DIR = os.path.join(MODEL_DIR,   "ablation")
COMBINED_DIR = os.path.join(MODEL_DIR,   "combined")
PRED_DIR     = os.path.join(PROJECT_DIR, "predictions")
RESULTS_DIR  = os.path.join(PROJECT_DIR, "results")

for d in [DATA_DIR, VOCAB_DIR, MODEL_DIR, ABLATION_DIR, COMBINED_DIR, PRED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

def get_splits(tb_name):
    folder = os.path.join(DATA_DIR, f"UD_Turkish-{tb_name}")
    files  = sorted(glob.glob(os.path.join(folder, "*.conllu")))
    splits = {}
    for f in files:
        name = os.path.basename(f)
        if   "train" in name: splits["train"] = f
        elif "dev"   in name: splits["dev"]   = f
        elif "test"  in name: splits["test"]  = f
    return splits

BOUN = get_splits("BOUN")
IMST = get_splits("IMST")
PUD  = get_splits("PUD")

# Prerequisites
A2B_PATH     = os.path.join(ABLATION_DIR, "a2b_lstm_with_pos.pt")
COMBINED_PT  = os.path.join(COMBINED_DIR, "combined_a2b_morph.pt")
COMBINED_STATE = COMBINED_PT + ".morph_state.pt"

if not os.path.exists(A2B_PATH):
    raise FileNotFoundError("a2b_lstm_with_pos.pt not found. Run 03_ablation.ipynb first.")
print(f"a2b_lstm_with_pos.pt : OK ({os.path.getsize(A2B_PATH):,} B)")
print(f"combined checkpoint  : {'exists' if os.path.exists(COMBINED_PT) else 'new — will train'}")


Mounted at /content/drive
a2b_lstm_with_pos.pt : OK (499,540,451 B)
combined checkpoint  : new — will train


---
## 1 — Install & Patch

In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "transformers==4.35.2" -q
!pip install supar -q
!pip install conllu -q

import os
PARSER_PY = "/usr/local/lib/python3.12/dist-packages/supar/parsers/parser.py"
OLD = "state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload))"
NEW = "state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload), weights_only=False)"
with open(PARSER_PY, "r") as f: src = f.read()
if OLD in src:
    with open(PARSER_PY, "w") as f: f.write(src.replace(OLD, NEW))
    print("Patch applied")
else:
    print("Patch already applied")

from transformers import AdamW, get_linear_schedule_with_warmup
import torch, transformers
print(f"transformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch runtime!'}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 43.6 MB/

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


transformers : 4.35.2
PyTorch      : 2.10.0+cu128
GPU          : NVIDIA A100-SXM4-40GB


---
## 2 — Hyperparameters

In [3]:
import torch, json

BERTURK_NAME = "dbmdz/bert-base-turkish-uncased"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

MORPH_DIM  = 32
MAX_FEATS  = 16

# LSTM-mode hyperparams — same as A2b in 03_ablation
BATCH_SIZE   = 5000
MAX_EPOCHS   = 30
PATIENCE     = 10
LR           = 2e-3
LR_MU        = 0.9
LR_NU        = 0.9
LR_EPS       = 1e-12
LR_WD        = 0.0
LR_DECAY     = 0.75
LR_DECAY_STEPS = 5000
GRAD_CLIP    = 5.0

SUPAR_EXTRA = dict(
    mbr=True, partial=False, tree=False,
    proj=False, punct=False, update_steps=1, clip=GRAD_CLIP
)

# Load feat vocab
feat_vocab_path = os.path.join(VOCAB_DIR, "feat_vocab.json")
with open(feat_vocab_path, "r", encoding="utf-8") as f:
    FEAT_VOCAB = json.load(f)
PAD_IDX    = FEAT_VOCAB["<PAD>"]
UNK_IDX    = FEAT_VOCAB["<UNK>"]
VOCAB_SIZE = len(FEAT_VOCAB)

print(f"LR={LR}  EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  BATCH={BATCH_SIZE}")
print(f"MORPH_DIM={MORPH_DIM}  VOCAB_SIZE={VOCAB_SIZE}  DEVICE={DEVICE}")


LR=0.002  EPOCHS=30  PATIENCE=10  BATCH=5000
MORPH_DIM=32  VOCAB_SIZE=87  DEVICE=cuda


---
## 3 — Morph Modules

Identical to 05a — `MorphEmbedder`, `MorphProjection`, `batch_to_morph_tensor`, `patch_model_forward`.
The patch works for LSTM-encoder mode: `encode()` returns BiLSTM output, morph is injected after.

In [4]:
import torch.nn as nn
import types

class MorphEmbedder(nn.Module):
    """Embedding table with sum pooling. Same as 05a."""
    def __init__(self, vocab_size, morph_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, morph_dim, padding_idx=pad_idx)
        nn.init.normal_(self.embedding.weight, std=0.01)
        with torch.no_grad():
            self.embedding.weight[pad_idx].fill_(0)
    def forward(self, morph_indices):
        return self.embedding(morph_indices).sum(dim=2)

class MorphProjection(nn.Module):
    """Projects [encoder_out || morph_emb] -> n_hidden. Same as 05a."""
    def __init__(self, n_hidden, morph_dim):
        super().__init__()
        self.proj = nn.Linear(n_hidden + morph_dim, n_hidden)
        nn.init.eye_(self.proj.weight[:n_hidden, :n_hidden])
        nn.init.zeros_(self.proj.weight[:, n_hidden:])
        nn.init.zeros_(self.proj.bias)
    def forward(self, enc_out, morph_emb):
        return self.proj(torch.cat([enc_out, morph_emb], dim=-1))

FEATS_COL = 5

def batch_to_morph_tensor(batch, feat_vocab, pad_idx, unk_idx, max_feats, device):
    batch_size = len(batch.sentences)
    seq_len    = max(len(s.values[FEATS_COL]) + 1 for s in batch.sentences)
    morph = torch.full((batch_size, seq_len, max_feats), pad_idx, dtype=torch.long)
    for b, sent in enumerate(batch.sentences):
        for t, feat_str in enumerate(sent.values[FEATS_COL], start=1):
            if feat_str == "_" or not feat_str: continue
            for f_idx, pair in enumerate(feat_str.split("|")[:max_feats]):
                morph[b, t, f_idx] = feat_vocab.get(pair.strip(), unk_idx)
    return morph.to(device)

def patch_model_forward(parser, morph_embedder, morph_projection):
    """
    Monkey-patch parser.model.forward() to inject morph features.
    Works for both BERT-encoder and LSTM-encoder modes:
    - BERT mode: encode() returns BERTurk subword representations
    - LSTM mode: encode() returns BiLSTM output on top of BERTurk
    Morph injection happens AFTER encode() in both cases.
    """
    from supar.utils.common import MIN
    _embedder    = morph_embedder
    _projection  = morph_projection
    _orig_encode = parser.model.encode

    def forward_with_morph(self, words, feats=None, morph_indices=None):
        x = _orig_encode(words, feats)
        if morph_indices is not None:
            x = _projection(x, _embedder(morph_indices))
        x = self.encoder_dropout(x)
        mask  = words.ne(self.args.pad_index) if len(words.shape) < 3                 else words.ne(self.args.pad_index).any(-1)
        arc_d = self.arc_mlp_d(x);  arc_h = self.arc_mlp_h(x)
        sib_s = self.sib_mlp_s(x);  sib_d = self.sib_mlp_d(x);  sib_h = self.sib_mlp_h(x)
        rel_d = self.rel_mlp_d(x);  rel_h = self.rel_mlp_h(x)
        s_arc = self.arc_attn(arc_d, arc_h).masked_fill_(~mask.unsqueeze(1), MIN)
        s_sib = self.sib_attn(sib_s, sib_d, sib_h).permute(0, 3, 1, 2)
        s_rel = self.rel_attn(rel_d, rel_h).permute(0, 2, 3, 1)
        return s_arc, s_sib, s_rel

    parser.model.forward = types.MethodType(forward_with_morph, parser.model)
    print("Model forward patched: morph injection active.")

print("Morph modules defined.")


Morph modules defined.


---
## 4 — Load Parser

Loads A2b checkpoint as starting point. Morph modules initialized fresh (or loaded from checkpoint if resuming).

In [12]:
from supar import CRF2oDependencyParser

# Load A2b checkpoint (BiLSTM + BERTurk + POS) as starting point
if os.path.exists(COMBINED_PT):
    print(f"Combined checkpoint found — loading: {COMBINED_PT}")
    parser = CRF2oDependencyParser.load(COMBINED_PT)
else:
    print(f"Loading A2b as starting point: {A2B_PATH}")
    parser = CRF2oDependencyParser.load(A2B_PATH)

parser.args.update(SUPAR_EXTRA)
parser.model = parser.model.to(DEVICE)
n_hidden = parser.model.arc_mlp_d.n_in
print(f"  n_hidden detected: {n_hidden}")

morph_embedder   = MorphEmbedder(VOCAB_SIZE, MORPH_DIM, PAD_IDX).to(DEVICE)
morph_projection = MorphProjection(n_hidden, MORPH_DIM).to(DEVICE)

# Load morph weights if resuming
if os.path.exists(COMBINED_STATE):
    state = torch.load(COMBINED_STATE, map_location=DEVICE, weights_only=False)
    morph_embedder.load_state_dict(state["morph_embedder"])
    morph_projection.load_state_dict(state["morph_projection"])
    print("Morph module weights loaded from checkpoint.")

patch_model_forward(parser, morph_embedder, morph_projection)

morph_params = (sum(p.numel() for p in morph_embedder.parameters()) +
                sum(p.numel() for p in morph_projection.parameters()))
total_params = sum(p.numel() for p in parser.model.parameters())
print(f"  encoder      : {parser.args.encoder}  (BiLSTM + BERTurk + POS)")
print(f"  n_hidden     : {n_hidden}")
print(f"  morph_params : {morph_params:,}")
print(f"  total_params : {total_params + morph_params:,}")
print(f"  device       : {DEVICE}")


Loading A2b as starting point: /content/drive/MyDrive/58T - Application Project/models/ablation/a2b_lstm_with_pos.pt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  n_hidden detected: 800
Model forward patched: morph injection active.
  encoder      : lstm  (BiLSTM + BERTurk + POS)
  n_hidden     : 800
  morph_params : 669,184
  total_params : 125,174,081
  device       : cuda


---
## 5 — Data Loaders & Optimizer

Adam + ExponentialLR — same as A2b training in 03_ablation.

In [13]:
from supar.utils import Dataset
from transformers import AdamW
from torch.optim.lr_scheduler import ExponentialLR

def make_loader(parser, path, batch_size, shuffle=True):
    dataset = Dataset(parser.transform, path)
    dataset.build(batch_size, 32)
    return dataset.loader

print(f"Building data loaders (batch_size={BATCH_SIZE})...")
train_loader = make_loader(parser, BOUN["train"], BATCH_SIZE, shuffle=True)
dev_loader   = make_loader(parser, BOUN["dev"],   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(parser, BOUN["test"],  BATCH_SIZE, shuffle=False)
pud_loader   = make_loader(parser, PUD["test"],   BATCH_SIZE, shuffle=False)
print(f"  train={len(train_loader)}  dev={len(dev_loader)}  test={len(test_loader)}  pud={len(pud_loader)} batches")

# LSTM-mode optimizer: Adam with ExponentialLR (same as A2b in 03_ablation)
# Include morph modules in the parameter set
all_params = (list(parser.model.parameters()) +
              list(morph_embedder.parameters()) +
              list(morph_projection.parameters()))

optimizer = torch.optim.Adam(
    [{"params": list(parser.model.parameters()), "lr": LR},
     {"params": list(morph_embedder.parameters()) +
                list(morph_projection.parameters()), "lr": LR}],
    betas=(LR_MU, LR_NU), eps=LR_EPS, weight_decay=LR_WD
)
scheduler = ExponentialLR(optimizer, gamma=LR_DECAY)
step_count = [0]  # for decay_steps tracking

print(f"  Optimizer: Adam (mu={LR_MU}, nu={LR_NU}, decay={LR_DECAY}, decay_steps={LR_DECAY_STEPS})")
print(f"  LR={LR}  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}")


Building data loaders (batch_size=5000)...
  train=40  dev=32  test=32  pud=32 batches
  Optimizer: Adam (mu=0.9, nu=0.9, decay=0.75, decay_steps=5000)
  LR=0.002  MAX_EPOCHS=30  PATIENCE=10


---
## 6 — Train Combined System

Fine-tunes A2b with morph injection. Expected runtime: ~2–3 hours on A100.
Skips training if checkpoint already exists.

In [14]:
import torch.nn as nn
from supar.utils.metric import AttachmentMetric

def run_epoch(parser, loader, morph_embedder, morph_projection,
              optimizer, scheduler, step_count, device, use_morph, training=True):
    if training:
        parser.model.train(); morph_embedder.train(); morph_projection.train()
    else:
        parser.model.eval(); morph_embedder.eval(); morph_projection.eval()

    total_loss = 0.0
    metric     = AttachmentMetric()
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for batch in loader:
            words, texts, *feats, arcs, sibs, rels = batch
            word_mask = words.ne(parser.args.pad_index)
            mask      = word_mask if len(words.shape) < 3 else word_mask.any(-1)
            mask[:, 0] = 0
            words = words.to(device); feats = [f.to(device) for f in feats]
            arcs  = arcs.to(device);  sibs  = sibs.to(device)
            rels  = rels.to(device);  mask  = mask.to(device)

            morph_indices = None
            if use_morph:
                morph_indices = batch_to_morph_tensor(
                    batch, FEAT_VOCAB, PAD_IDX, UNK_IDX, MAX_FEATS, device)
                morph_indices = morph_indices[:, :words.shape[1], :]

            s_arc, s_sib, s_rel = parser.model(words, feats, morph_indices)
            loss, s_arc = parser.model.loss(
                s_arc, s_sib, s_rel, arcs, sibs, rels, mask,
                parser.args.mbr, parser.args.partial)

            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(all_params, GRAD_CLIP)
                optimizer.step()
                step_count[0] += 1
                if step_count[0] % LR_DECAY_STEPS == 0:
                    scheduler.step()

            arc_preds, rel_preds = parser.model.decode(
                s_arc, s_sib, s_rel, mask,
                parser.args.tree, parser.args.mbr, parser.args.proj)
            total_loss += loss.item()
            metric(arc_preds, rel_preds, arcs, rels, mask)

    return total_loss / len(loader), metric


if os.path.exists(COMBINED_PT) and os.path.exists(COMBINED_STATE):
    print("Combined checkpoint exists — skipping training.")
    print("Delete combined_a2b_morph.pt and .morph_state.pt to retrain.")
else:
    print(f"Training: A2b (BiLSTM+BERTurk+POS) + UD FEATS morph injection")
    print(f"Epochs={MAX_EPOCHS}  Patience={PATIENCE}  LR={LR}  Device={DEVICE}")
    print("=" * 65)

    best_las = 0.0; patience_ct = 0; history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, _ = run_epoch(
            parser, train_loader, morph_embedder, morph_projection,
            optimizer, scheduler, step_count, DEVICE, use_morph=True, training=True)
        dev_loss, dev_m = run_epoch(
            parser, dev_loader, morph_embedder, morph_projection,
            optimizer, scheduler, step_count, DEVICE, use_morph=True, training=False)
        uas = dev_m.uas * 100; las = dev_m.las * 100
        print(f"Epoch {epoch:>2}/{MAX_EPOCHS}  train {train_loss:.4f} | dev {dev_loss:.4f} | UAS {uas:.2f} LAS {las:.2f}", end="")

        if las > best_las:
            best_las = las; patience_ct = 0
            torch.save({"morph_embedder": morph_embedder.state_dict(),
                        "morph_projection": morph_projection.state_dict()},
                       COMBINED_STATE)
            parser.save(COMBINED_PT)
            print(" (saved)", end="")
        else:
            patience_ct += 1
            print(f" (patience {patience_ct}/{PATIENCE})", end="")
        print()
        history.append({"epoch": epoch, "train_loss": train_loss,
                        "dev_loss": dev_loss, "dev_uas": uas, "dev_las": las})
        if patience_ct >= PATIENCE:
            print(f"Early stopping at epoch {epoch}."); break

    print(f"\nBest dev LAS: {best_las:.2f}")
    with open(os.path.join(RESULTS_DIR, "combined_history.json"), "w") as f:
        json.dump(history, f, indent=2)
    print("History saved.")


Training: A2b (BiLSTM+BERTurk+POS) + UD FEATS morph injection
Epochs=30  Patience=10  LR=0.002  Device=cuda
Epoch  1/30  train 1.0370 | dev 1.0561 | UAS 81.22 LAS 74.06 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:152: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  h = apply_permutation(hx[0], permutation)
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:153: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  c = apply_permutation(hx[1], permutation)


Epoch  2/30  train 0.8323 | dev 1.0160 | UAS 80.84 LAS 73.39 (patience 1/10)
Epoch  3/30  train -0.5075 | dev 0.9991 | UAS 81.62 LAS 74.12 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:152: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  h = apply_permutation(hx[0], permutation)
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:153: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  c = apply_permutation(hx[1], permutation)


Epoch  4/30  train 10.2366 | dev 0.9732 | UAS 81.48 LAS 74.25 (saved)


...

Epoch  9/30  train -87.5476 | dev 0.9575 | UAS 81.17 LAS 74.13 (patience 1/10)
Epoch 10/30  train -112.4494 | dev 0.9790 | UAS 81.19 LAS 74.08 (patience 2/10)
Epoch 11/30  train -30.0603 | dev 0.9688 | UAS 81.41 LAS 74.41 (patience 3/10)
Epoch 12/30  train -35.3579 | dev 0.9736 | UAS 82.15 LAS 75.17 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:152: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  h = apply_permutation(hx[0], permutation)
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:153: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  c = apply_permutation(hx[1], permutation)


Epoch 13/30  train -147.7230 | dev 0.9419 | UAS 81.55 LAS 74.31 (patience 1/10)
Epoch 14/30  train -285.1167 | dev 0.9925 | UAS 81.11 LAS 73.99 (patience 2/10)
Epoch 15/30  train -4.0308 | dev 0.9656 | UAS 81.56 LAS 74.33 (patience 3/10)
Epoch 16/30  train -541.9363 | dev 0.9352 | UAS 81.69 LAS 74.38 (patience 4/10)
Epoch 17/30  train -275.4255 | dev 0.9411 | UAS 81.75 LAS 74.68 (patience 5/10)
Epoch 18/30  train -1230.4851 | dev 0.9429 | UAS 82.09 LAS 74.90 (patience 6/10)
Epoch 19/30  train -5.9559 | dev 0.9840 | UAS 81.76 LAS 74.67 (patience 7/10)
Epoch 20/30  train -655.0290 | dev 0.9730 | UAS 81.27 LAS 73.85 (patience 8/10)
Epoch 21/30  train 6.7162 | dev 0.9591 | UAS 81.69 LAS 74.45 (patience 9/10)
Epoch 22/30  train -20.4974 | dev 0.9505 | UAS 82.31 LAS 75.21 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:152: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  h = apply_permutation(hx[0], permutation)
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:153: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  c = apply_permutation(hx[1], permutation)


Epoch 23/30  train -6.3150 | dev 1.0078 | UAS 81.65 LAS 74.35 (patience 1/10)
Epoch 24/30  train -91.1238 | dev 0.9616 | UAS 82.35 LAS 75.14 (patience 2/10)
Epoch 25/30  train -434.4072 | dev 0.9478 | UAS 81.92 LAS 74.73 (patience 3/10)
Epoch 26/30  train -2674.5660 | dev 0.9272 | UAS 81.48 LAS 74.47 (patience 4/10)
Epoch 27/30  train -97.7966 | dev 0.9882 | UAS 82.14 LAS 75.01 (patience 5/10)
Epoch 28/30  train -203.7422 | dev 0.9717 | UAS 81.88 LAS 74.94 (patience 6/10)
Epoch 29/30  train 321.4110 | dev 0.9436 | UAS 81.80 LAS 74.95 (patience 7/10)
Epoch 30/30  train -2423.9416 | dev 0.9646 | UAS 81.69 LAS 74.64 (patience 8/10)

Best dev LAS: 75.21
History saved.


---
## 7 — Evaluate

Evaluates on BOUN test and PUD zero-shot.
Per-relation LAS computed from the training loop (morph-injected forward) — not from parser.predict().

In [17]:
import json, numpy as np
import conllu as conllu_lib
from collections import defaultdict

# Load best morph weights
if os.path.exists(COMBINED_STATE):
    state = torch.load(COMBINED_STATE, map_location=DEVICE, weights_only=False)
    morph_embedder.load_state_dict(state["morph_embedder"])
    morph_projection.load_state_dict(state["morph_projection"])
    print("Best combined morph weights loaded.")

def per_relation_las(gold_sents, pred_sents):
    counts = defaultdict(lambda: [0, 0])
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        for g, p in zip(gt, pt):
            counts[g["deprel"]][1] += 1
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]:
                counts[g["deprel"]][0] += 1
    return {rel: {"correct": c, "total": t,
                  "LAS": round(100 * c / t, 2) if t else 0.0}
            for rel, (c, t) in sorted(counts.items(), key=lambda x: -x[1][1])}

def evaluate_loader(parser, loader, morph_embedder, morph_projection,
                    device, use_morph, label):
    _, metric = run_epoch(parser, loader, morph_embedder, morph_projection,
                           None, None, [0], device, use_morph=use_morph, training=False)
    result = {"system": label,
              "UAS": round(metric.uas * 100, 2),
              "LAS": round(metric.las * 100, 2)}
    print(f"  {label:<50} UAS {result['UAS']:>6.2f}   LAS {result['LAS']:>6.2f}")
    return result

results = []
print("=" * 70)

# A2b alone (starting point — no morph)
r = evaluate_loader(parser, test_loader, morph_embedder, morph_projection,
                    DEVICE, use_morph=False, label="A2b alone (BiLSTM+BERTurk+POS, no morph)")
r["test"] = "BOUN"; results.append(r)

r = evaluate_loader(parser, pud_loader, morph_embedder, morph_projection,
                    DEVICE, use_morph=False, label="A2b alone -> PUD (zero-shot)")
r["test"] = "PUD"; results.append(r)

# Combined: A2b + UD FEATS morph injection
r = evaluate_loader(parser, test_loader, morph_embedder, morph_projection,
                    DEVICE, use_morph=True, label="A2b + UD FEATS morph (combined)  ★")
r["test"] = "BOUN"; results.append(r)

r = evaluate_loader(parser, pud_loader, morph_embedder, morph_projection,
                    DEVICE, use_morph=True, label="A2b + UD FEATS morph -> PUD (zero-shot)")
r["test"] = "PUD"; results.append(r)

# Save per-sentence npy for McNemar's tests
for sys_name, use_morph_flag, test_name, conllu_path in [
        ("combined_a2b_morph", True,  "BOUN", BOUN["test"]),
        ("combined_a2b_morph", True,  "PUD",  PUD["test"])]:
    pred_conllu = os.path.join(PRED_DIR, f"{sys_name}_{test_name}.conllu")
    parser.predict(conllu_path, pred=pred_conllu, verbose=False)
    with open(conllu_path, encoding="utf-8") as f:
        gold_sents = conllu_lib.parse(f.read())
    with open(pred_conllu, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())
    sent_las = [
        sum(1 for g, p in zip(
            [t for t in gold if isinstance(t["id"], int)],
            [t for t in pred if isinstance(t["id"], int)])
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]) ==
        len([t for t in gold if isinstance(t["id"], int)])
        for gold, pred in zip(gold_sents, pred_sents)
    ]
    np.save(os.path.join(PRED_DIR, f"{sys_name}_{test_name}_sent_las.npy"),
            np.array(sent_las, dtype=bool))
    print(f"  Saved: {sys_name}_{test_name}.conllu + _sent_las.npy")

# Per-relation LAS — use the existing test_loader (built in cell 5 with gold labels)
print("\nComputing per-relation LAS (morph-injected, from training loop)...")
parser.model.eval(); morph_embedder.eval(); morph_projection.eval()

counts   = defaultdict(lambda: [0, 0])
sent_idx = 0

with open(BOUN["test"], encoding="utf-8") as f:
    gold_sents_all = conllu_lib.parse(f.read())

# Build rel vocab lookup once
rel_vocab_lookup = None
for field in parser.transform.flattened_fields:
    if hasattr(field, "vocab") and hasattr(field, "name") \
            and field.name in ("rels", "rel", "REL", "deprel"):
        rv = field.vocab
        if hasattr(rv, "itos"):
            rel_vocab_lookup = rv.itos
        elif isinstance(rv, dict):
            rel_vocab_lookup = {v: k for k, v in rv.items()}
        break

with torch.no_grad():
    for batch in test_loader:   # ← test_loader from cell 5, has gold labels
        words, texts, *feats, arcs, sibs, rels = batch
        word_mask = words.ne(parser.args.pad_index)
        mask = word_mask if len(words.shape) < 3 else word_mask.any(-1)
        mask[:, 0] = 0
        words = words.to(DEVICE); feats = [f.to(DEVICE) for f in feats]
        arcs  = arcs.to(DEVICE);  sibs  = sibs.to(DEVICE)
        rels  = rels.to(DEVICE);  mask  = mask.to(DEVICE)

        morph_indices = batch_to_morph_tensor(
            batch, FEAT_VOCAB, PAD_IDX, UNK_IDX, MAX_FEATS, DEVICE)
        morph_indices = morph_indices[:, :words.shape[1], :]

        s_arc, s_sib, s_rel = parser.model(words, feats, morph_indices)
        arc_preds, rel_preds = parser.model.decode(
            s_arc, s_sib, s_rel, mask,
            parser.args.tree, parser.args.mbr, parser.args.proj)
        arc_preds = arc_preds.cpu()
        rel_preds = rel_preds.cpu()
        mask_cpu  = mask.cpu()

        for i in range(arc_preds.shape[0]):
            if sent_idx >= len(gold_sents_all): break
            sent      = gold_sents_all[sent_idx]
            real_toks = [t for t in sent if isinstance(t["id"], int)]
            valid_len = int(mask_cpu[i].sum().item())
            for j in range(min(valid_len, len(real_toks))):
                gold_rel = real_toks[j]["deprel"]
                gold_arc = real_toks[j]["head"]
                pred_arc = int(arc_preds[i, j+1].item())
                pred_rel = rel_vocab_lookup[int(rel_preds[i, j+1].item())]
                counts[gold_rel][1] += 1
                if pred_arc == gold_arc and pred_rel == gold_rel:
                    counts[gold_rel][0] += 1
            sent_idx += 1

breakdown = {rel: {"correct": c, "total": t,
                   "LAS": round(100*c/t, 2) if t else 0.0}
             for rel, (c, t) in sorted(counts.items(), key=lambda x: -x[1][1])}

total   = sum(v["total"]   for v in breakdown.values())
correct = sum(v["correct"] for v in breakdown.values())
print(f"  Overall LAS from per-rel loop: {100*correct/total:.2f}  (should be ~73.54)")

br_path = os.path.join(RESULTS_DIR, "combined_a2b_morph_BOUN_per_rel.json")
with open(br_path, "w") as f:
    json.dump(breakdown, f, indent=2)
print(f"  Per-relation saved: {br_path}")

results_path = os.path.join(RESULTS_DIR, "combined_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {results_path}")

Best combined morph weights loaded.


ValueError: not enough values to unpack (expected at least 5, got 4)

---
## 8 — Results Table

In [ ]:
# Load previous best systems for comparison
import json

prev_results = {}
for phase_file, label in [
    ("ablation_results.json",  "ablation"),
    ("adapter_results.json",   "adapters"),
    ("morph_v2_results.json",  "morph_v2"),
]:
    path = os.path.join(RESULTS_DIR, phase_file)
    if os.path.exists(path):
        with open(path) as f:
            for r in json.load(f):
                prev_results[(r["system"], r["test"])] = r["LAS"]

# Reference LAS values for comparison
REF = {
    "02 Baseline (BOUN)"          : 61.96,
    "03 A2b BiLSTM+BERTurk+POS"   : prev_results.get(("a2b", "BOUN"), 74.73),
    "04 Adapter BOUN"              : prev_results.get(("adapter_boun", "BOUN"), 68.74),
    "05a Morph V2 UD FEATS"        : prev_results.get(("V2: BERTurk + UD FEATS", "BOUN"),
                                      prev_results.get(("morph_v2_ud_feats", "BOUN"), 71.79)),
    "Özateş et al. (2022) SOTA"   : 82.10,
}

print("=" * 72)
print("  COMBINED SYSTEM RESULTS vs Individual Systems")
print("  A2b (BiLSTM + BERTurk + POS) + UD FEATS Morphological Injection")
print("=" * 72)

print(f"\n  {'System':<52} {'Test':<6} {'UAS':>7} {'LAS':>7}")
print(f"  {'-'*70}")
for r in results:
    print(f"  {r['system']:<52} {r['test']:<6} {r['UAS']:>7.2f} {r['LAS']:>7.2f}")

print(f"\n  {'Reference systems (BOUN test)':}")
print(f"  {'-'*40}")
for sys_name, las in REF.items():
    print(f"  {sys_name:<40} LAS {las:.2f}")

boun_combined = next((r for r in results if "combined" in r["system"] and r["test"]=="BOUN"), None)
if boun_combined:
    print(f"\n  Delta vs A2b alone    : ΔLAS {boun_combined['LAS'] - REF['03 A2b BiLSTM+BERTurk+POS']:+.2f}")
    print(f"  Delta vs Morph V2     : ΔLAS {boun_combined['LAS'] - REF['05a Morph V2 UD FEATS']:+.2f}")
    print(f"  Delta vs Baseline     : ΔLAS {boun_combined['LAS'] - REF['02 Baseline (BOUN)']:+.2f}")
    print(f"  Gap to SOTA (Özateş)  : ΔLAS {boun_combined['LAS'] - REF['Özateş et al. (2022) SOTA']:+.2f}")


---
## 9 — Per-Relation LAS Breakdown

Compares Combined system against Baseline, A2b, and Adapter on BOUN test.

In [ ]:
import json

# Load per-relation for comparison with previous best systems
per_rel_files = {
    "Baseline"       : "baseline_boun_BOUN_per_rel.json",
    "A2b"            : "a2b_BOUN_per_rel.json",
    "Adapter"        : "adapter_boun_BOUN_per_rel.json",
    "Combined ★"     : "combined_a2b_morph_BOUN_per_rel.json",
}

per_rel = {}
for label, filename in per_rel_files.items():
    path = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(path):
        with open(path) as f:
            per_rel[label] = json.load(f)
        print(f"  Loaded {label}: {len(per_rel[label])} relations")
    else:
        print(f"  MISSING: {label} ({filename})")

if per_rel:
    ref      = per_rel.get("Baseline", next(iter(per_rel.values())))
    all_rels = sorted(ref.keys(), key=lambda r: ref[r]["total"], reverse=True)
    available  = [k for k in per_rel_files.keys() if k in per_rel]
    col_w = 13

    print()
    print("Per-Relation LAS — BOUN Test Set — Top 25 Relations")
    print()
    header = f"  {'Relation':<14} {'Count':>6}" + "".join(f"  {l[:col_w]:>{col_w}}" for l in available)
    print(header)
    print("  " + "-" * (len(header) - 2))

    for rel in all_rels[:25]:
        count = ref.get(rel, {}).get("total", 0)
        row   = f"  {rel:<14} {count:>6}"
        for k in available:
            v = per_rel[k].get(rel, {}).get("LAS", 0.0)
            row += f"  {v:>{col_w}.2f}"
        print(row)

    # Save merged
    merged = {}
    for rel in all_rels:
        merged[rel] = {"total": ref.get(rel, {}).get("total", 0)}
        for k in available:
            merged[rel][k] = per_rel[k].get(rel, {}).get("LAS", 0.0)
    with open(os.path.join(RESULTS_DIR, "combined_per_rel_comparison.json"), "w") as f:
        json.dump(merged, f, indent=2)
    print(f"\nPer-relation comparison saved.")


---
## 10 — Verify Saved Files

In [ ]:
expected = [
    (COMBINED_DIR, "combined_a2b_morph.pt"),
    (COMBINED_DIR, "combined_a2b_morph.pt.morph_state.pt"),
    (PRED_DIR,     "combined_a2b_morph_BOUN.conllu"),
    (PRED_DIR,     "combined_a2b_morph_BOUN_sent_las.npy"),
    (PRED_DIR,     "combined_a2b_morph_PUD.conllu"),
    (PRED_DIR,     "combined_a2b_morph_PUD_sent_las.npy"),
    (RESULTS_DIR,  "combined_results.json"),
    (RESULTS_DIR,  "combined_history.json"),
    (RESULTS_DIR,  "combined_a2b_morph_BOUN_per_rel.json"),
    (RESULTS_DIR,  "combined_per_rel_comparison.json"),
]

print(f"{'Status':<8} {'Size':>12}  {'Dir':<14} File")
print("-" * 80)
all_ok = True
for directory, filename in expected:
    path   = os.path.join(directory, filename)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path):,} B" if exists else "-"
    status = "OK" if exists else "MISSING"
    folder = os.path.basename(directory)
    if not exists: all_ok = False
    print(f"  {status:<6} {size:>12}  {folder:<14} {filename}")
print()
if all_ok:
    print("All files present. 07_combined complete.")
else:
    print("Some files missing — rerun the relevant sections above.")
